# 依照張棚集的測站順序建立測站list


In [8]:
import re

# 讀取文字檔 (請確保檔案編碼正確，通常為 utf-8)
with open('測站資料與位置.txt', 'r', encoding='utf-8') as file:
    content = file.read()

# 使用正規表達式找出所有 'Station:' 後面的一組代碼
station_codes = re.findall(r'Station:\s+([A-Z0-9]+)', content)

# 印出結果
print(len(station_codes))

32


In [2]:
import pandas as pd
import re
def filter_seismic_metadata(input_path, output_path, target_event_id, station_list):
    """|
    從 Metadata 中篩選特定事件 ID 與測站清單的資料並存檔
    """
    print(f"🔍 正在讀取原始檔案: {input_path}")
    
    # 1. 讀取 CSV
    # 加上 dtype={...} 可以確保 ID 不會因為太長被科學記號化
    df = pd.read_csv(input_path, low_memory=False)

    # 2. 執行條件篩選
    # 條件 A: source_event_id 符合
    # 條件 B: station_code 在 station_list 內
    # 注意: 如果 CSV 裡的 ID 是字串，target_event_id 也要傳字串
    filtered_df = df[
        (df["source_event_id"].astype(str) == str(target_event_id)) & 
        (df["station_code"].isin(station_list))
    ]

    # 3. 檢查結果並儲存
    if filtered_df.empty:
        print("⚠️ 篩選完成：找不到符合條件的資料，請檢查 ID 或測站名稱是否正確。")
    else:
        filtered_df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"✅ 篩選完成！共 {len(filtered_df)} 筆資料。")
        print(f"💾 結果已儲存至: {output_path}")

if __name__ == "__main__":
    # --- 參數設定 ---
    input_csv = r"Y:\CWA_processed_data\all_metadata.csv"  # 您的原始 Metadata 路徑
    output_csv = "filtered_event_stations.csv"            # 輸出的檔名
    
    # 目標 Event ID
    target_id = 2112310800
    
    # 您的測站清單 (請根據需求修改)
    my_station_list = ["HWA", "EHY", "TWD", "SKL"] 
    # 讀取文字檔 (請確保檔案編碼正確，通常為 utf-8)
    with open('測站資料與位置.txt', 'r', encoding='utf-8') as file:
        content = file.read()

    # 使用正規表達式找出所有 'Station:' 後面的一組代碼
    my_station_list = re.findall(r'Station:\s+([A-Z0-9]+)', content)

    # 執行函數
    filter_seismic_metadata(input_csv, output_csv, target_id, my_station_list)

🔍 正在讀取原始檔案: Y:\CWA_processed_data\all_metadata.csv
⚠️ 篩選完成：找不到符合條件的資料，請檢查 ID 或測站名稱是否正確。


In [3]:
import h5py
import os

# --- 設定路徑與條件 ---
source_file = r'Y:\CWA_processed_data\all.hdf5'  # 原始大檔案
output_file = 'new_data.hdf5'                    # 要輸出的新檔案
target_prefix = '2021_2112310800_'               # 篩選條件 (開頭字串)

def extract_specific_event(src_path, dst_path, prefix):
    print(f"[*] 準備從 {src_path} 提取資料...")
    
    # 檢查原始檔案是否存在
    if not os.path.exists(src_path):
        print(f"[!] 錯誤：找不到來源檔案！請檢查路徑或網路磁碟機。")
        return

    try:
        # 同時開啟兩個檔案：來源檔唯讀 ('r')，目標檔寫入 ('w')
        # 'w' 模式會建立新檔案，如果檔案已存在則會被覆寫
        with h5py.File(src_path, 'r') as src, h5py.File(dst_path, 'w') as dst:
            
            print(f"[+] 檔案開啟成功，開始尋找以 '{prefix}' 開頭的資料...")
            copied_count = 0
            
            # 遍歷原始檔案根目錄下的所有 Key
            for key in src.keys():
                # 如果名稱符合我們要的開頭
                if key.startswith(prefix):
                    # 將這個 Dataset 或 Group 完整複製到新檔案
                    # src.copy(要複製的物件名稱, 目的地檔案)
                    src.copy(key, dst)
                    print(f"✅ 已複製: {key}")
                    copied_count += 1
            
            print("-" * 40)
            if copied_count > 0:
                print(f"[+] 提取大功告成！共複製了 {copied_count} 筆資料到 '{dst_path}'")
            else:
                print(f"[!] 警告：在第一層結構中，找不到任何以 '{prefix}' 開頭的資料。")
                print("    (如果您的資料被包在子資料夾(Group)內，需要修改為遞迴搜尋)")

    except Exception as e:
        print(f"[!] 處理檔案時發生錯誤: {e}")

if __name__ == "__main__":
    # 執行提取任務
    extract_specific_event(source_file, output_file, target_prefix)

[*] 準備從 Y:\CWA_processed_data\all.hdf5 提取資料...
[+] 檔案開啟成功，開始尋找以 '2021_2112310800_' 開頭的資料...
✅ 已複製: 2021_2112310800_ALS_HL_BB_10
✅ 已複製: 2021_2112310800_ALS_HL_SMT_10
✅ 已複製: 2021_2112310800_CHKH_HL_BH_10
✅ 已複製: 2021_2112310800_CHK_HL_SMT_10
✅ 已複製: 2021_2112310800_CHN4_HL_SMT_10
✅ 已複製: 2021_2112310800_CHN5_HL_SMT_10
✅ 已複製: 2021_2112310800_DPDB_HL_BB_10
✅ 已複製: 2021_2112310800_EAH_HL_BH_10
✅ 已複製: 2021_2112310800_EAS_HL_SMT_10
✅ 已複製: 2021_2112310800_ECB_HL_SMT_10
✅ 已複製: 2021_2112310800_ECL_HL_SMT_10
✅ 已複製: 2021_2112310800_EGC_HL_SMT_10
✅ 已複製: 2021_2112310800_EGFH_HL_BH_10
✅ 已複製: 2021_2112310800_EGS_HL_SMT_10
✅ 已複製: 2021_2112310800_EHYH_HL_BH_10
✅ 已複製: 2021_2112310800_EHY_HL_SMT_10
✅ 已複製: 2021_2112310800_ENA_HL_SMT_10
✅ 已複製: 2021_2112310800_ENT_HL_SMT_10
✅ 已複製: 2021_2112310800_ESL_HL_SMT_10
✅ 已複製: 2021_2112310800_ETLH_HL_BH_10
✅ 已複製: 2021_2112310800_ETL_HL_SMT_10
✅ 已複製: 2021_2112310800_ETM_HL_SMT_10
✅ 已複製: 2021_2112310800_EWT_HL_BH_10
✅ 已複製: 2021_2112310800_FULB_HL_BB_10
✅ 已複製: 2021_211231080

In [14]:
import pandas as pd
import os

def find_min_p_arrival(csv_path):
    """
    讀取 CSV 檔案並找出 trace_p_arrival_sample 欄位的最小值
    """
    print(f"🔍 正在讀取檔案: {csv_path}")
    
    if not os.path.exists(csv_path):
        print("[!] 錯誤：找不到指定的 CSV 檔案，請檢查路徑。")
        return

    try:
        # 1. 讀取 CSV 檔案
        df = pd.read_csv(csv_path, low_memory=False)
        
        # 2. 檢查欄位是否存在
        if "trace_p_arrival_sample" not in df.columns:
            print("[!] 錯誤：CSV 檔案中找不到 'trace_p_arrival_sample' 欄位。")
            return
            
        # 3. 取得該欄位的最小值
        min_value = df["trace_p_arrival_sample"].min()
        
        print("-" * 30)
        print(f"✅ 尋找成功！")
        print(f"📉 'trace_p_arrival_sample' 的最小值為: {min_value}")
        print("-" * 30)
        
        return min_value

    except Exception as e:
        print(f"[!] 讀取或處理檔案時發生錯誤: {e}")

if __name__ == "__main__":
    # --- 請替換成您實際的 CSV 檔案路徑 ---
    # 假設是您之前使用的 metadata 檔案
    my_csv_file = r"filtered_event_stations.csv" 
    
    # 執行函數
    find_min_p_arrival(my_csv_file)

🔍 正在讀取檔案: filtered_event_stations.csv
------------------------------
✅ 尋找成功！
📉 'trace_p_arrival_sample' 的最小值為: 2732
------------------------------


In [16]:
import pandas as pd
import os

def get_station_codes(csv_path):
    """
    讀取 CSV 檔案並提取 station_code 欄位的所有值存成變數
    """
    print(f"🔍 正在讀取檔案: {csv_path}")
    
    if not os.path.exists(csv_path):
        print("[!] 錯誤：找不到指定的 CSV 檔案，請檢查路徑。")
        return None, None

    try:
        # 1. 讀取 CSV 檔案
        df = pd.read_csv(csv_path, low_memory=False)
        
        # 2. 檢查欄位是否存在
        if "station_code" not in df.columns:
            print("[!] 錯誤：CSV 檔案中找不到 'station_code' 欄位。")
            return None, None
            
        # ---------------------------------------------------------
        # 3. 提取所有的值，並存成新的變數 (轉換成 Python 的 List)
        # ---------------------------------------------------------
        
        # 變數 A：包含所有值 (可能會有重複的測站)
        all_stations = df["station_code"].tolist()
        
        # 變數 B：去除重複值 (通常比較實用，可以得到乾淨的測站清單)
        # .dropna() 是為了把可能為空的欄位剔除
        unique_stations = df["station_code"].dropna().unique().tolist()
        
        print("-" * 40)
        print(f"✅ 提取成功！")
        print(f"📊 總資料筆數: {len(all_stations)} 筆")
        print(f"🌟 不重複的測站數量: {len(unique_stations)} 個")
        print("-" * 40)
        
        # 回傳這兩個變數供後續使用
        return all_stations, unique_stations

    except Exception as e:
        print(f"[!] 讀取或處理檔案時發生錯誤: {e}")
        return None, None

if __name__ == "__main__":
    # --- 請替換成您實際的 CSV 檔案路徑 ---
    my_csv_file = r"filtered_event_stations.csv" 
    
    # 執行函數，並將結果存入新的變數中
    all_stations_list, unique_stations_list = get_station_codes(my_csv_file)
    
    # 這裡可以印出前 10 個不重複的測站來檢查看看
    if unique_stations_list:
        print(f"\n前 10 個不重複的測站代碼：\n{unique_stations_list[:]}")

🔍 正在讀取檔案: filtered_event_stations.csv
----------------------------------------
✅ 提取成功！
📊 總資料筆數: 21 筆
🌟 不重複的測站數量: 19 個
----------------------------------------

前 10 個不重複的測站代碼：
['TWB1', 'NWF', 'TWA', 'NHY', 'NWL', 'TAP', 'ZUZH', 'TWS1', 'NTS', 'ETL', 'ETLH', 'TWD', 'EYL', 'ETM', 'SHUL', 'ESL', 'EHYH', 'EHY', 'FULB']


In [1]:
import h5py
import os

# --- 設定路徑與條件 ---
source_file = r'Y:\CWA_processed_data\all.hdf5'  # 原始大檔案
output_file = 'new_data.hdf5'                    # 要輸出的新檔案
target_prefix = '2021_2112310800_'               # 篩選條件 (開頭字串)

# --- 新增：您的測站清單 ---
station_list = ['TWB1', 'NWF', 'TWA', 'NHY', 'NWL', 'TAP', 'ZUZH', 'TWS1', 
                'NTS', 'ETL', 'ETLH', 'TWD', 'EYL', 'ETM', 'SHUL', 'ESL', 
                'EHYH', 'EHY', 'FULB']

def extract_specific_event_and_stations(src_path, dst_path, prefix, stations):
    print(f"[*] 準備從 {src_path} 提取資料...")
    
    # 檢查原始檔案是否存在
    if not os.path.exists(src_path):
        print(f"[!] 錯誤：找不到來源檔案！請檢查路徑或網路磁碟機。")
        return

    try:
        # 同時開啟兩個檔案：來源檔唯讀 ('r')，目標檔寫入 ('w')
        with h5py.File(src_path, 'r') as src, h5py.File(dst_path, 'w') as dst:
            
            print(f"[+] 檔案開啟成功，開始尋找符合【事件前綴】與【測站清單】的資料...")
            copied_count = 0
            
            # 遍歷原始檔案根目錄下的所有 Key
            for key in src.keys():
                
                # 條件一：名稱符合我們要的開頭
                if key.startswith(prefix):
                    
                    # 將 Key 用底線切割 (例如: ['2021', '2112310800', 'ALS', 'HL', 'BB', '10'])
                    parts = key.split('_')
                    
                    # 確保切割後的長度足夠，避免發生 Index 錯誤
                    if len(parts) > 2:
                        # 條件二：提取出的測站代碼必須在清單中
                        station_code = parts[2] 
                        
                        if station_code in stations:
                            # 兩者皆符合，執行複製
                            src.copy(key, dst)
                            print(f"✅ 已複製: {key}")
                            copied_count += 1
            
            print("-" * 40)
            if copied_count > 0:
                print(f"[+] 提取大功告成！共複製了 {copied_count} 筆資料到 '{dst_path}'")
            else:
                print(f"[!] 警告：找不到任何同時符合事件與測站清單的資料。")

    except Exception as e:
        print(f"[!] 處理檔案時發生錯誤: {e}")

if __name__ == "__main__":
    # 執行提取任務
    extract_specific_event_and_stations(source_file, output_file, target_prefix, station_list)

[*] 準備從 Y:\CWA_processed_data\all.hdf5 提取資料...
[+] 檔案開啟成功，開始尋找符合【事件前綴】與【測站清單】的資料...
✅ 已複製: 2021_2112310800_EHYH_HL_BH_10
✅ 已複製: 2021_2112310800_EHY_HL_SMT_10
✅ 已複製: 2021_2112310800_ESL_HL_SMT_10
✅ 已複製: 2021_2112310800_ETLH_HL_BH_10
✅ 已複製: 2021_2112310800_ETL_HL_SMT_10
✅ 已複製: 2021_2112310800_ETM_HL_SMT_10
✅ 已複製: 2021_2112310800_FULB_HL_BB_10
✅ 已複製: 2021_2112310800_SHUL_HL_BH_10
✅ 已複製: 2021_2112310800_TWB1_HL_BB_10
✅ 已複製: 2021_2112310800_TWB1_HL_SMT_10
✅ 已複製: 2021_2112310800_TWD_HL_SMT_10
----------------------------------------
[+] 提取大功告成！共複製了 11 筆資料到 'new_data.hdf5'
